# Ge 158 Problem Set 3, Problem 1: starter notebook
Python version of `ge158_2026_pset3_p1_starter.js`. It does the same calculations; use whichever you prefer.

**Setup.** This notebook runs in Google Colab (File > Open notebook > Upload), or in Jupyter with `earthengine-api` and `geemap` installed. Set `PROJECT` to the Google Cloud project you used for Problem Set 2. The first time you run the setup cell, it asks you to sign in to your Google account.

In [ ]:
# If geemap is missing (e.g., in a local Jupyter), uncomment the next line
# %pip install -q earthengine-api geemap
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = 'your-cloud-project-id'  # the Cloud project from Problem Set 2
ee.Authenticate()
ee.Initialize(project=PROJECT)

**Helpers.** SRTM elevation (m) and functions that sample an image (`y`) and the elevation (`z_km`) at random points, return them as a pandas DataFrame, and plot `y` versus `z_km`.

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003')

def sample_vs_elevation(img, region, scale):
    fc = (dem.divide(1000).rename('z_km').addBands(img.rename('y'))
          .sample(region=region, scale=scale, numPixels=3000,
                  geometries=False, tileScale=4))
    return pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])

def plot_vs_elevation(df, title, ylabel):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df['z_km'], df['y'], '.', ms=2)
    ax.set_xlabel('Elevation z (km)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.show()

## Part (a): ERA5-Land monthly mean surface pressure (Pa → hPa)

In [ ]:
tibet = ee.Geometry.Rectangle([78, 26, 92, 36])
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').select('surface_pressure')
p_jan = era5.filterDate('2023-01-01', '2023-02-01').first().divide(100)
p_samples = sample_vs_elevation(p_jan.log(), tibet, 11132)
plot_vs_elevation(p_samples, 'Surface pressure, Tibet, Jan 2023', 'ln(P / 1 hPa)')
# Least-squares fit y = intercept + slope * z_km. Note that the slope is
# per km; it is not a scale height:
# slope, intercept = np.polyfit(p_samples['z_km'], p_samples['y'], 1)

## Part (b): MODIS column water vapor
Column water vapor (× 0.001 → cm), clear-sky retrievals only (over clouds, the product reports the water vapor above the cloud), averaged over one month.

In [ ]:
nepal = ee.Geometry.Rectangle([82, 26.5, 88.5, 30])

def clear_sky_wv(img):
    cloud_mask = img.select('AOD_QA').bitwiseAnd(7)  # bits 0-2
    return img.select('Column_WV').updateMask(cloud_mask.eq(1))

wv = (ee.ImageCollection('MODIS/061/MCD19A2_GRANULES')
      .filterBounds(nepal).filterDate('2023-11-01', '2023-12-01')
      .map(clear_sky_wv).mean().multiply(0.001))
w_samples = sample_vs_elevation(wv.log(), nepal, 2000)
plot_vs_elevation(w_samples, 'Column water vapor, Nepal, Nov 2023', 'ln(w_vap / 1 cm)')

## Part (e): Sentinel-2 top-of-atmosphere reflectance near Mt. Everest
3 Dec 2023, a mosaic of all tiles from that overpass. Use the layer control (top right of the map) to switch between bands.

In [ ]:
everest = ee.Geometry.Rectangle([86.5, 27.0, 87.3, 28.15])
toa = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
       .filterBounds(everest).filterDate('2023-12-03', '2023-12-04')
       .mosaic().divide(10000))  # reflectance
Map = geemap.Map()
Map.centerObject(everest, 9)
Map.addLayer(toa, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.4}, 'True color')
Map.addLayer(toa.select('B8A'), {'min': 0, 'max': 0.5}, 'B8A (865 nm)')
Map.addLayer(toa.select('B9'), {'min': 0, 'max': 0.3}, 'B9 (945 nm)')
Map.addLayer(toa.select('B10'), {'min': 0, 'max': 0.05}, 'B10 (1375 nm)')
Map

In [ ]:
ln_ratio = toa.select('B9').divide(toa.select('B8A')).log()
plot_vs_elevation(sample_vs_elevation(ln_ratio, everest, 300),
                  'Sentinel-2 near Mt. Everest', 'ln(B9 / B8A)')
plot_vs_elevation(sample_vs_elevation(toa.select('B10'), everest, 300),
                  'Sentinel-2 near Mt. Everest', 'B10 reflectance')